---
title: "📈 Inferno Mutual Information Between Predictands and Predictors"
author: "Maksim Ohvrill"
date: "5/1/2025"
format:
  html:
    toc: true
    toc-location: right
    code-fold: true
bibliography: references.bib
csl: ieee-with-url.csl
jupyter: R
---

This notebook analyzes the mutual information between patient predictors (CNN logits, Age, Gender, etc.) and lung condition predictands (effusion and atelectasis). It demonstrates that CNN logits carry nearly all predictive power, while auxiliary information adds little beyond what the model already captures.


In [1]:
# -------------------------------
# Setup
# -------------------------------

# Set working directory one level up
setwd("..")
library(inferno)

# Load reusable utilities
source("RScripts/reusableUtils.R")

# Define general output directory
output_dir <- "data/plots"
if (!dir.exists(output_dir)) {
  dir.create(output_dir)
}

# Define model and data paths
learnt_dir <- "data/inferno/combinedML50"
setup <- load_metadata_testdata(learnt_dir)
metadata <- setup$metadata
test_data <- setup$testdata

## Mutual Information Analysis

### 📖 Overview

Mutual Information (MI) quantifies the dependency between two random variables.  
In this study, MI measures how much information about:

- Patient **age**  
- CNN **logit outputs** (raw scores before applying sigmoid)

explains:

- The ground truth labels: **LABEL_EFFUSION** and **LABEL_ATELECTASIS**.

### 📈 Mutual Information Between CNN Logits and Age

The first analysis computes the mutual information between CNN logits and patient age.

- **Inputs**:  
  $Y_1 = \{\text{LOGIT\_ATELECTASIS}, \text{LOGIT\_EFFUSION}\}$  
  $Y_2 = \{\text{AGE}\}$

- **Formula**:

$$
MI(Y_1, Y_2) = \sum_{y_1, y_2} p(y_1, y_2) \log\left( \frac{p(y_1, y_2)}{p(y_1)p(y_2)} \right)
$$

- **Results Reported**:
  - Mutual Information ($MI$)
  - Conditional entropies: $H(Y_1|Y_2)$ and $H(Y_2|Y_1)$
  - Marginal entropies: $H(Y_1)$ and $H(Y_2)$
  - Maximum achievable MI: $MI_{\text{max}} = H(Y_1)$

### 🧪 Mutual Information Between Labels and Predictors

Mutual information is calculated between:

- $Y_1 = \{\text{LABEL\_ATELECTASIS}, \text{LABEL\_EFFUSION}\}$
- $Y_2 = \{\text{AGE, GENDER, VP, LOGIT\_EFFUSION, LOGIT\_ATELECTASIS}\}$

This measures how well patient metadata and CNN predictions explain the ground truth labels.

### 🔍 Isolation of Age and Logit Contributions

To isolate contributions:

1. **Without Age**:  
   $Y_2 = \{\text{GENDER, VP, LOGIT\_EFFUSION, LOGIT\_ATELECTASIS}\}$

2. **Without Logits**:  
   $Y_2 = \{\text{AGE, GENDER, VP}\}$

Comparing mutual information values shows how much each group adds uniquely.

### 📏 Results Interpretation

- **Absolute Differences**:

For age contribution:

$$
\Delta_{\text{age}} = MI_{\text{full}} - MI_{\text{no age}}
$$

For logits contribution:

$$
\Delta_{\text{logits}} = MI_{\text{full}} - MI_{\text{no logits}}
$$

- **Relative Importance**:

Relative contribution of logits:

$$
\text{Relative Importance of Logits} = \frac{\Delta_{\text{logits}}}{MI_{\text{full}}} \times 100
$$

In [ ]:
# -------------------------------
# Mutual Information Calculations
# -------------------------------

# Mutual information between logits and age
mi_logits_age <- calculate_mi(
  Y1names = c("LOGIT_ATELECTASIS", "LOGIT_EFFUSION"),
  Y2names = c("AGE"),
  learntdir = learnt_dir
)

# Mutual information between predictands and all predictors
predictands <- c("LABEL_ATELECTASIS", "LABEL_EFFUSION")
predictors <- setdiff(metadata$name, predictands)
mi_full <- calculate_mi(
  Y1names = predictands,
  Y2names = predictors,
  learntdir = learnt_dir
)

# Mutual information without age
mi_no_age <- calculate_mi(
  Y1names = predictands,
  Y2names = setdiff(predictors, "AGE"),
  learntdir = learnt_dir
)

# Mutual information without logits
mi_no_logits <- calculate_mi(
  Y1names = predictands,
  Y2names = setdiff(predictors, c("LOGIT_EFFUSION", "LOGIT_ATELECTASIS")),
  learntdir = learnt_dir
)



## Logits vs Age ##
    Metric    Value    Error
1       MI 0.102233 0.009189
2 CondEn12 5.403046 0.022729
3 CondEn21 5.941547 0.016973
4      En1 5.505221 0.021819
5      En2 6.043806 0.016413
6    MImax 5.505221 0.021819

## Predictands vs All Predictors ##
    Metric     Value    Error
1       MI  0.348822 0.015973
2 CondEn12  1.438250 0.018778
3 CondEn21 13.012228 0.032607
4      En1  1.786676 0.012833
5      En2 13.361043 0.028460
6    MImax  1.786676 0.012833

## Predictands vs Predictors (No Age) ##
    Metric    Value    Error
1       MI 0.372488 0.015273
2 CondEn12 1.409627 0.018428
3 CondEn21 7.037614 0.027041
4      En1 1.782179 0.012904
5      En2 7.410116 0.022711
6    MImax 1.782179 0.012904

## Predictands vs Predictors (No Logits) ##
    Metric    Value    Error
1       MI 0.023712 0.004761
2 CondEn12 1.775833 0.013480
3 CondEn21 7.952449 0.017689
4      En1 1.799432 0.012677
5      En2 7.976187 0.017282
6    MImax 1.799432 0.012677

## Differences and Relative Importa

In [8]:
# -------------------------------
# Organize Results in Tables
# -------------------------------

# Helper to format and print MI results
format_mi_table <- function(mi_result, label) {
  cat("\n", label, "\n")
  print(data.frame(
    Metric = c("MI", "CondEn12", "CondEn21", "En1", "En2", "MImax"),
    Value = round(c(
      mi_result$MI['value'], mi_result$CondEn12['value'],
      mi_result$CondEn21['value'], mi_result$En1['value'],
      mi_result$En2['value'], mi_result$MImax['value']
    ), 6),
    Error = round(c(
      mi_result$MI['error'], mi_result$CondEn12['error'],
      mi_result$CondEn21['error'], mi_result$En1['error'],
      mi_result$En2['error'], mi_result$MImax['error']
    ), 6)
  ))
}

format_mi_table(mi_logits_age, "Logits vs Age")
format_mi_table(mi_full, "Predictands vs All Predictors")
format_mi_table(mi_no_age, "Predictands vs Predictors (No Age)")
format_mi_table(mi_no_logits, "Predictands vs Predictors (No Logits)")

# -------------------------------
# Compare Mutual Information
# -------------------------------

mi_diff_age <- mi_full$MI['value'] - mi_no_age$MI['value']
mi_diff_logits <- mi_full$MI['value'] - mi_no_logits$MI['value']
relative_importance_logits <- 100 * mi_diff_logits / mi_full$MI['value']

cat("\nDifferences and Relative Importance\n")
comparison_table <- data.frame(
  Comparison = c("Difference due to Age", "Difference due to Logits", "Relative Importance of Logits"),
  Value = round(c(mi_diff_age, mi_diff_logits, relative_importance_logits), 6)
)
print(comparison_table)


 Logits vs Age 
    Metric    Value    Error
1       MI 0.102233 0.009189
2 CondEn12 5.403046 0.022729
3 CondEn21 5.941547 0.016973
4      En1 5.505221 0.021819
5      En2 6.043806 0.016413
6    MImax 5.505221 0.021819

 Predictands vs All Predictors 
    Metric     Value    Error
1       MI  0.348822 0.015973
2 CondEn12  1.438250 0.018778
3 CondEn21 13.012228 0.032607
4      En1  1.786676 0.012833
5      En2 13.361043 0.028460
6    MImax  1.786676 0.012833

 Predictands vs Predictors (No Age) 
    Metric    Value    Error
1       MI 0.372488 0.015273
2 CondEn12 1.409627 0.018428
3 CondEn21 7.037614 0.027041
4      En1 1.782179 0.012904
5      En2 7.410116 0.022711
6    MImax 1.782179 0.012904

 Predictands vs Predictors (No Logits) 
    Metric    Value    Error
1       MI 0.023712 0.004761
2 CondEn12 1.775833 0.013480
3 CondEn21 7.952449 0.017689
4      En1 1.799432 0.012677
5      En2 7.976187 0.017282
6    MImax 1.799432 0.012677

Differences and Relative Importance
               

## 🔢 Mutual Information Results

### Logits vs Age

| Metric    | Value   | Error   |
|:----------|:--------|:--------|
| MI        | 0.102233 | 0.009189 |
| CondEn12  | 5.403046 | 0.022729 |
| CondEn21  | 5.941547 | 0.016973 |
| En1       | 5.505221 | 0.021819 |
| En2       | 6.043806 | 0.016413 |
| MImax     | 5.505221 | 0.021819 |

### Predictands vs All Predictors

| Metric    | Value   | Error   |
|:----------|:--------|:--------|
| MI        | 0.348822 | 0.015973 |
| CondEn12  | 1.438250 | 0.018778 |
| CondEn21  | 13.012228 | 0.032607 |
| En1       | 1.786676 | 0.012833 |
| En2       | 13.361043 | 0.028460 |
| MImax     | 1.786676 | 0.012833 |

### Predictands vs Predictors (No Age)

| Metric    | Value   | Error   |
|:----------|:--------|:--------|
| MI        | 0.372488 | 0.015273 |
| CondEn12  | 1.409627 | 0.018428 |
| CondEn21  | 7.037614 | 0.027041 |
| En1       | 1.782179 | 0.012904 |
| En2       | 7.410116 | 0.022711 |
| MImax     | 1.782179 | 0.012904 |

### Predictands vs Predictors (No Logits)

| Metric    | Value   | Error   |
|:----------|:--------|:--------|
| MI        | 0.023712 | 0.004761 |
| CondEn12  | 1.775833 | 0.013480 |
| CondEn21  | 7.952449 | 0.017689 |
| En1       | 1.799432 | 0.012677 |
| En2       | 7.976187 | 0.017282 |
| MImax     | 1.799432 | 0.012677 |

### Differences and Relative Importance

| Comparison                  | Value    |
|:-----------------------------|:---------|
| Difference due to Age        | -0.023666 |
| Difference due to Logits     | 0.325110 |
| Relative Importance of Logits | 93.202234 |


## 🎯 Summary of Key Findings

- **Logits capture almost all predictive information**:  
  - Predictands (effusion and atelectasis) are highly dependent on CNN logits.  
  - Removing logits causes mutual information (MI) to drop from $0.3488$ to $0.0237$.  
  - This corresponds to a **loss of $0.3251$**, or **$93.2\%$** of the total predictive information.

- **Age contributes negligibly**:  
  - Removing Age slightly *increases* MI (from $0.3488$ to $0.3725$), but the change is within numerical error.  
  - Difference due to Age is small: **$-0.0237$**.  
  - Age alone explains very little about the lung condition labels beyond what logits already encode.

- **CNN logits dominate predictive capability**:  
  - Neural network logits already capture Age-related effects.  
  - Adding extra auxiliary data like Age, Gender, or Ventilation Pressure adds almost no extra predictive value.

### 🧠 Interpretations and Information Theory Insights

**General Predictive Power**:  
- The mutual information between all predictors and predictands is about $0.35$ Shannon ($\text{Sh}$), indicating modest predictive ability.  
- The maximum Shannon entropy of the predictands is about $1.79\, \text{Sh}$.  
- This means that most uncertainty about the true lung condition remains even after using all available predictors.

**Logits vs Age Dependence**:  
- The mutual information between the two logits and Age is only $0.10\, \text{Sh}$, out of a maximum of about $5.5\, \text{Sh}$.  
- Knowing Age reduces uncertainty about logits only slightly, from about $39$ binary guesses to $37$.  
- This agrees with the flatness of the conditional probability plots against Age.

**Critical Role of Logits**:  
- CNN logits carry almost all usable information for predicting lung conditions.  
- Without logits, prediction ability nearly disappears, showing a $93\%$ loss in mutual information.

**Strength of Bayesian Inference**:  
- CNN logits are strong but not perfectly reliable.  
- Bayesian recalibration (Inferno) adjusts patient-specific probabilities, improving flexibility and handling uncertainty much better than a hard threshold.

### 🛠️ Practical Implications

**For clinical models**:  
- Decision-making should be based mainly on CNN logits, calibrated with Bayesian methods.  
- Relying only on hard thresholds loses important case-by-case details.

**For future improvements**:  
- Simply adding auxiliary data (like Age or Gender) does not help much.  
- Major gains would require new types of patient information that reveal different risk factors not already captured by the CNN logits, such as:  
  - **Geography**:  
    Regional health differences, such as higher rates of lung conditions in certain areas due to differences in healthcare access, climate, or socioeconomic status.
  - **Environmental exposure**:  
    Information about air pollution levels, occupational exposures, or residential proximity to industrial zones could explain variability in lung conditions like effusion or atelectasis.
  - **Hospital-specific practices**:  
    Differences in ventilation protocols, surgical techniques, or diagnostic standards between hospitals might influence the true prevalence and detection of lung complications.
  - **Pre-treatment factors**:  
    Data about patient interventions before imaging, such as oxygen therapy, ventilation support, or recent surgical procedures, could affect how lung pathology presents and is interpreted.
  - **Underlying health conditions**:  
    Broader comorbidity profiles (such as chronic heart failure or recent infections) might interact with the likelihood of developing effusion or atelectasis, introducing variability not captured by age or gender alone.